# ALPSS benchmark on real files

Runs the current branch on every real file in the Girder collection **ALPSS Benchmark Dataset**
and reports each result's error against the file's results file.

A case is a pair `<id>.csv` (the scope trace) + `<id>-results.csv` (the reference) in the
collection's `real` folder. Upload a new pair there and it is part of the next run.

Needs `GIRDER_API_URL` and `GIRDER_API_KEY` in the environment, and the kernel's Python must have
this branch's `alpss` installed (`poetry install`). Run the cells from top to bottom.

## Settings

In [ ]:
import os
import sys

BENCH_DIR = os.path.abspath("")  # this notebook sits in benchmarks/
CONFIG = os.path.join(BENCH_DIR, "benchmark_config.json")  # one config for every file
LABEL = "my-branch"  # results go to benchmarks/runs/<LABEL>/
BASELINE = None  # e.g. "main": a previous run's label to compare against
SET = []  # config overrides for every file, e.g. ["start_time_user=cusum"]

COLLECTION_ID = "6b057dccb00526ec98a4da67"  # ALPSS Benchmark Dataset
REAL_DIR = os.path.join(BENCH_DIR, "data", "real")

## Fetch

Downloads the pairs missing from `benchmarks/data/real/`; files already there are kept.

In [ ]:
import json
import urllib.parse
import urllib.request

def girder(path, raw=False, method="GET", **params):
    url = f"{os.environ['GIRDER_API_URL']}/{path}?{urllib.parse.urlencode(params)}"
    req = urllib.request.Request(url, method=method)
    if path != "api_key/token":
        req.add_header("Girder-Token", TOKEN)
    data = urllib.request.urlopen(req).read()
    return data if raw else json.loads(data)

TOKEN = girder("api_key/token", method="POST", key=os.environ["GIRDER_API_KEY"])["authToken"]["token"]
folder, = girder("folder", parentType="collection", parentId=COLLECTION_ID, name="real")

os.makedirs(REAL_DIR, exist_ok=True)
for item in girder("item", folderId=folder["_id"], limit=0):
    path = os.path.join(REAL_DIR, item["name"])
    if os.path.exists(path):
        continue
    print("downloading", item["name"])
    with open(path, "wb") as f:
        f.write(girder(f"item/{item['_id']}/download", raw=True))
print(len(os.listdir(REAL_DIR)), "files in", REAL_DIR)

## Run

In [ ]:
import subprocess

def bench(*args):
    """Run a benchmarks/ script; show its output only if it fails."""
    done = subprocess.run([sys.executable, *args], cwd=BENCH_DIR, capture_output=True, text=True)
    if done.returncode:
        print(done.stdout[-3000:], done.stderr[-3000:])
        raise RuntimeError(f"{args[0]} failed")

cmd = ["run.py", "--real", "--label", LABEL, "--config", CONFIG]
if SET:
    cmd += ["--set", *SET]
bench(*cmd)
with open(os.path.join(BENCH_DIR, "runs", LABEL, "run.json")) as f:
    run = json.load(f)
for case_id, status in run["cases"].items():
    print(f"{case_id:30s} {status['status']}  {status['error'] or ''}")

## Report

Per metric: how many files measured it and the spread of their errors (measured − reference).

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display

cmd = ["summarize.py", os.path.join("runs", LABEL)]
if BASELINE:
    cmd += ["--against", os.path.join("runs", BASELINE)]
bench(*cmd)

run_dir = os.path.join(BENCH_DIR, "runs", LABEL)
display(pd.read_csv(os.path.join(run_dir, "stats.csv")))
display(Image(os.path.join(run_dir, "errors.png")))

Per file, worst first, for each metric: where the branch does well and where it does not.

In [ ]:
summary = pd.read_csv(os.path.join(run_dir, "summary.csv"))
summary["abs_error"] = summary["error"].abs()
for metric, rows in summary.groupby("metric", sort=False):
    display(Markdown(f"**{metric}**"))
    display(rows.sort_values("abs_error", ascending=False)
                [["case", "truth", "measured", "error", "rel_error"]]
                .rename(columns={"truth": "reference"}))

In [ ]:
if BASELINE:
    name = os.path.basename(os.path.normpath(BASELINE))
    display(pd.read_csv(os.path.join(run_dir, f"comparison_vs_{name}.csv")))